# Tamil Whisper Finetuning

Prepare paired audio/transcripts and fine-tune a Tamil Whisper model with a WER metric.

Stage: project experiment.


## Preparing Dataset
    * Checking the audio files and thier corresponding text files exists or not
    * Creating a csv with the existing pairs
    * Train and test splitting (using 20% for test)
    * Creating csv files for train and test

In [ ]:
import os

In [ ]:
!pwd

In [ ]:
from huggingface_hub import login
use_auth_token = os.environ.get("HF_TOKEN")  # create and add the token from hugging face
login(use_auth_token)

In [ ]:
import os
import pandas as pd
import chardet

# Define the root directory
root_directory = "/Training"

# Initialize empty lists
file_paths = []
transcripts = []

# Iterate through the files in the audio_folder
audio_folder = os.path.join(root_directory, "Audio")
transcript_folder = os.path.join(root_directory, "Transcripts")

for audio_filename in os.listdir(audio_folder):
    audio_file_path = os.path.join(audio_folder, audio_filename)
    transcript_filename = audio_filename.replace(".wav", ".txt")
    transcript_file_path = os.path.join(transcript_folder, transcript_filename)

    # Debug prints to check the paths and filenames
    print(f"Audio file path: {audio_file_path}")
    print(f"Transcript file path: {transcript_file_path}")

    # Check if the corresponding transcript file exists
    if os.path.exists(transcript_file_path):
        # Check if the audio file and transcript file names match
        if transcript_filename == os.path.basename(transcript_file_path):
            file_paths.append(audio_file_path)
            try:
                with open(transcript_file_path, 'rb') as file:
                  rawdata = file.read()
                  result = chardet.detect(rawdata)
                  encoding = result['encoding']
                  print(f"Detected encoding: {encoding}")
                with open(transcript_file_path, 'r', encoding=encoding) as transcript_file:
                    transcript = transcript_file.read()
                transcripts.append(transcript)
            except UnicodeDecodeError:
                print(f"Error decoding file: {transcript_filename}")

        else:
            print(f"File names do not match for {audio_filename}")
    else:
        print(f"Transcript not found for {audio_filename}")

# Create a DataFrame if there are audio-transcript pairs
if len(file_paths) > 0:
    data = {'path': file_paths, 'transcript': transcripts}
    df = pd.DataFrame(data)

    # Display the DataFrame
    print(df)
else:
    print("No audio-transcript pairs found.")


In [ ]:
df

In [ ]:
df['path'][1]

In [ ]:
# creating training and testing csv files
from sklearn.model_selection import train_test_split

df = pd.read_csv("dataset.csv")

# Split the DataFrame into train and test sets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Save the train and test DataFrames to separate CSV files
train_df.to_csv("train_data.csv", index=False)
test_df.to_csv("test_data.csv", index=False)

# Print the shapes of the train and test DataFrames
print("Train data shape:", train_df.shape)
print("Test data shape:", test_df.shape)

## Whisper Model

## Pre-Processing

In [ ]:
from datasets import Dataset
from datasets import Audio
import gc
import pandas as pd

# loading the csv files
train_df = pd.read_csv("train_data.csv")
test_df = pd.read_csv("test_data.csv")

print(len(test_df))
train_df

In [ ]:
# creating the dataset using the class methods Dataset.from_pandas()
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)
train_dataset[0]

In [ ]:
# create arrays of each audio file and append those values as a column in the above datasets.
# To do this we will use the cast_column function from Dataset.
train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16000))
test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16000))

### Prepare Feature Extractor, Tokenizer and Data
### Load WhisperFeatureExtractor and Tokenizer

In [ ]:
# !pip install transformers

In [ ]:
# !pip install --upgrade datasets transformers accelerate soundfile librosa evaluate jiwer tensorboard gradio

In [ ]:
## import feature extractor and tokenizer
from transformers import WhisperFeatureExtractor
from transformers import WhisperTokenizer

feature_extractor = WhisperFeatureExtractor.from_pretrained("vasista22/whisper-tamil-medium")
tokenizer = WhisperTokenizer.from_pretrained("vasista22/whisper-tamil-medium",task="transcribe")

In [ ]:
## Combine To Create A WhisperProcessor

from transformers import WhisperProcessor
processor = WhisperProcessor.from_pretrained("vasista22/whisper-tamil-medium", task="transcribe")

### Preparing data for the model
    * generating log-mel spectrogram
    * creating label_ids using tokenizer

In [ ]:
train_dataset[0]

In [ ]:
input_str = train_dataset[0]["transcript"]
labels = tokenizer(input_str).input_ids
decoded_with_special = tokenizer.decode(labels, skip_special_tokens=False)
decoded_str = tokenizer.decode(labels, skip_special_tokens=True)

print(f"Input:                 {input_str}")
print(f"Decoded w/ special:    {decoded_with_special}")
print(f"Decoded w/out special: {decoded_str}")
print(f"Are equal:             {input_str == decoded_str}")


In [ ]:
def prepare_dataset(examples):
    # compute log-Mel input features from input audio array
    audio = examples["audio"]
    examples["input_features"] = feature_extractor(
        audio["array"], sampling_rate=16000).input_features[0]

    del examples["audio"]

    sentences = examples["transcript"]
    
    # encode target text to label ids
    examples["labels"] = tokenizer(sentences).input_ids

    del examples["transcript"]
    return examples

#### Using the dataset’s.map method, we can apply the data preparation function to each of our training examples

In [ ]:
train_dataset = train_dataset.map(prepare_dataset, num_proc=2)
test_dataset = test_dataset.map(prepare_dataset, num_proc=2)

### Training and Evaluation
    * Defining a data-collator to handle input data and label spearetly
    * The word error rate (WER) metric will be used to measure the effectiveness of the model. A compute metrics function must be created to handle this calculation.
    * Loading a pre-trained checkpoint: A pre-trained checkpoint must be loaded and properly set up for training.
    * Defining the training arguments: The trainer will use these to create the training schedule.

In [ ]:
train_dataset

In [ ]:
## data-collator
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
## Evalution metric
import evaluate
metric = evaluate.load("wer")

In [ ]:
# evaluation metric function
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    
    return {"wer": wer}


In [ ]:
# Loading a Pre-Trained Checkpoint
from transformers import WhisperForConditionalGeneration
model = WhisperForConditionalGeneration.from_pretrained("vasista22/whisper-tamil-medium")

In [ ]:
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./pretrainedwhisper-medium-native-v3",  # change to a repo name of your choice
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=200,
    max_steps=2000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=500,
    eval_steps=500,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
    optim="adamw_bnb_8bit"
)

In [ ]:
## training
from transformers import Seq2SeqTrainer
# early_stopping_callback = EarlyStoppingCallback(early_stopping_threshold=0.001, early_stopping_patience=2)
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    # callbacks=[early_stopping_callback]
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained(training_args.output_dir)
processor.save_pretrained(training_args.output_dir)